# Inspect Driver — vLLM Provider Demo

Same surface as the HF-backed `Inspect_Bridge_Demo`, but routes through `tl_bridge_vllm`
— vLLM under the hood instead of `AutoModelForCausalLM`. The point is dataset-scale
evals: vLLM's PagedAttention + continuous batching lets parallel `inspect_eval` samples
stream through a single resident model, where the HF provider serializes them.

**Boundaries served (vLLM-specific):**
- ✅ `blocks.{i}.hook_out` (resid_post)
- ✅ `blocks.{i}.attn.hook_out` (attn_out)
- ✅ `blocks.{i}.mlp.hook_out` (mlp_out)
- ⛔ `blocks.{i}.hook_in` (resid_pre) — vLLM fuses the block-input read into the next
  layer; no Python-visible hook point.
- ⛔ `blocks.{i}.ln2.hook_in` (resid_mid) — derived from resid_pre + attn_out.
- (Always non-fireable) head-split `hook_q/k/v/z`, `hook_pattern`, `embed`, `ln_final`,
  `unembed`.

Use `boot_inspect(provider="tl_bridge")` for the gated boundaries (small models, CPU,
exact parity). Use `boot_inspect(provider="tl_bridge_vllm")` for throughput on the
served boundaries.

**Prerequisites:** an NVIDIA GPU with enough memory for the chosen model in fp16 +
vLLM's KV cache (≈4–8 GB for `meta-llama/Llama-3.2-1B`). vLLM is GPU-only; this
notebook will not run on CPU.

## Setup

Install TransformerLens with both `inspect` and vLLM:

```bash
pip install "transformer_lens[inspect]" vllm
```

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import torch

assert torch.cuda.is_available(), "vLLM is GPU-only; this notebook needs CUDA."

from transformer_lens.model_bridge.remote_bridge import RemoteBridge
from transformer_lens.model_bridge.transformer_bridge import TransformerBridge

MODEL = "meta-llama/Llama-3.2-1B"
PROMPT = "The quick brown fox"
torch.manual_seed(0)

## Step 1 — Boot the model through Inspect (vLLM provider)

`boot_inspect(..., provider="tl_bridge_vllm")` constructs the same `RemoteBridge` you
get from the HF provider; the swap is invisible to downstream code. The provider's
structural self-check enumerates which boundary kinds vLLM can serve; the bridge profile
is filtered to that set, and any gated boundary surfaces as a UserWarning.

First boot takes ~30–60s (vLLM loads weights, profiles KV cache, captures CUDA graphs).

In [ ]:
bridge = RemoteBridge.boot_inspect(MODEL, provider="tl_bridge_vllm")
print("bridge:", type(bridge).__name__)
# UserWarning surfaces on boot when boundaries are gated; print here for visibility.
print("capability note:", bridge._driver._model.api.capability_note())

## Step 2 — Listing available hooks

`bridge.hook_dict` is provider-agnostic. The fireable set vs the HF provider differs by
the gated kinds (resid_pre, resid_mid); everything else is identical.

In [ ]:
print("hook_dict entries:", len(bridge.hook_dict))
fireable = sorted(bridge._driver.supported_hook_points)
print("fireable (native):", len(fireable))
print("layer 0:", [h for h in fireable if h.startswith("blocks.0.")])
print("non-fireable (use boot_transformers):", len(bridge._driver.non_fireable_hook_points))
# resid_pre / resid_mid live in non-fireable for this provider:
print("layer 0 non-fireable:", [h for h in sorted(bridge._driver.non_fireable_hook_points)
                                if h.startswith("blocks.0.")])

## Step 3 — `run_with_cache` over the Inspect+vLLM boundary

Activations cross the Inspect boundary as the same wire envelope the HF provider emits,
get converted to torch at the bridge boundary, and surface as a normal `ActivationCache`.

In [ ]:
tokens = bridge.to_tokens(PROMPT)
logits, cache = bridge.run_with_cache(tokens)
print("logits:", tuple(logits.shape))
for hk in ["blocks.0.hook_out", "blocks.0.attn.hook_out", "blocks.0.mlp.hook_out"]:
    print(f"{hk}: shape={tuple(cache[hk].shape)} dtype={cache[hk].dtype}")
nxt = int(logits[0, -1].argmax())
print("next-token:", nxt, repr(bridge.tokenizer.decode([nxt])))

## Step 4 — Parity vs `boot_transformers`

vLLM runs the same weights as HF, but in fp16 with fused kernels (PagedAttention,
QKVParallelLinear). The residual stream matches to **fp16 tolerance** — looser than the
HF provider's exact match, tight enough that next-token argmax agrees on the served
boundaries.

In [ ]:
hf = TransformerBridge.boot_transformers(MODEL, dtype=torch.float16)
toks = hf.to_tokens(PROMPT)  # shared, BOS-prepended

hf_logits, hf_cache = hf.run_with_cache(toks)
i_logits, i_cache = bridge.run_with_cache(toks)

print("argmax match:", int(hf_logits[0, -1].argmax()) == int(i_logits[0, -1].argmax()))
# resid_post / attn_out / mlp_out across a sampling of layers — vLLM's served kinds.
for hk in ["blocks.0.hook_out", "blocks.0.attn.hook_out", "blocks.0.mlp.hook_out",
           "blocks.8.hook_out", "blocks.15.hook_out"]:
    a, b = hf_cache[hk].float(), i_cache[hk].float()
    rel = (a - b).norm() / (a.norm() + 1e-6)
    print(f"{hk}: rel_L2={rel.item():.2e}  maxdiff={(a - b).abs().max().item():.2e}")
del hf  # free HF before vLLM keeps going

## Step 5 — Interventions

Full affine vocabulary (`suppress` / `scale` / `add` / `set`) flows through to the vLLM
worker via the per-hook scale+bias buffer swap. Same surface as the HF provider — the
intervention dict is identical.

In [ ]:
clean = int(bridge.forward(toks)[0, -1].argmax())

hk = "blocks.0.hook_out"  # resid_post
supp_logits, supp_cache = bridge.run_with_cache(
    toks, intervene={hk: {"op": "suppress"}}
)
suppressed = int(supp_logits[0, -1].argmax())
print(f"{hk} |max| after suppress:", supp_cache[hk].abs().max().item())
print(f"argmax: clean={clean}  suppressed={suppressed}  changed={clean != suppressed}")

revert = int(bridge.forward(toks)[0, -1].argmax())
print(f"revert={revert}  matches clean={revert == clean}")

# Tear down vLLM before the eval-pipeline section boots its own. vLLM has no
# LLM.shutdown(); the driver's close() destroys the distributed env so the next
# Inspect-launched LLM(...) starts clean.
bridge.close()

## Inspect-native workflows

`tl_bridge_vllm/<model>` is also a normal `inspect_ai` model — the eval pipeline boots
vLLM, runs samples through it (parallel, batched), and reads activations off
`ModelOutput.metadata`. The next cells show the same four patterns as the HF demo:
**(6)** eval-native generation with logprobs + usage, **(7)** capture_activations
alongside a scored eval, **(8)** per-turn capture across a rollout, **(9)** tool-aware
generation. The helpers are provider-agnostic (they read the same wire envelope), so the
cells are essentially the HF cells with the model string swapped.

In [ ]:
# (6) Eval-native generation: tl_bridge_vllm is a normal Inspect model — real multi-token
# generation with token logprobs + usage from vLLM's sampler.
import tempfile

from inspect_ai import Task
from inspect_ai import eval as inspect_eval
from inspect_ai.dataset import Sample
from inspect_ai.solver import generate

run = tempfile.mkdtemp()
gen_log = inspect_eval(
    Task(dataset=[Sample(input="The capital of France is")], solver=generate()),
    model=f"tl_bridge_vllm/{MODEL}",
    max_tokens=6,
    logprobs=True,
    top_logprobs=3,
    log_dir=f"{run}/gen",
    display="none",
)[0]
out = gen_log.samples[0].output
print("completion:", repr(out.completion))
print("usage:", out.usage.input_tokens, "->", out.usage.output_tokens, "tokens")
top3 = out.choices[0].logprobs.content[0].top_logprobs
print("1st-token top-3:", [(t.token, round(t.logprob, 2)) for t in top3])

In [ ]:
# (7) Capture activations alongside a scored eval. Same helpers as the HF demo — they
# read metadata['activations'] which both providers emit in the same wire envelope.
import pathlib

from inspect_ai.analysis import SampleSummary, samples_df
from inspect_ai.scorer import includes

from transformer_lens.model_bridge.sources.inspect import activations_column, capture_activations

cap_task = Task(
    dataset=[
        Sample(input="The capital of France is", target="Paris"),
        Sample(input="The opposite of hot is", target="cold"),
    ],
    solver=[capture_activations(["blocks.8.hook_out"], output_dir=f"{run}/acts"), generate()],
    scorer=includes(),
)
cap_log = inspect_eval(
    cap_task, model=f"tl_bridge_vllm/{MODEL}", max_tokens=4, log_dir=f"{run}/cap", display="none"
)[0]
print("status:", cap_log.status, "| npz:", [p.name for p in pathlib.Path(f"{run}/acts").glob("*.npz")])
df = samples_df(f"{run}/cap", columns=[*SampleSummary, activations_column()])
print(df[["id", "tl_activations"]].to_string(index=False))

In [ ]:
# (8) Per-turn capture during a multi-token generate. vLLM's compiled hooks overwrite
# buffer row 0 on each decode step, so the provider takes a single-token snapshot of the
# prompt activations BEFORE the eval generate (extra ~prompt prefill — small relative to
# typical multi-token completions).
from transformer_lens.model_bridge.sources.inspect import turn_activations

turn_log = inspect_eval(
    Task(dataset=[Sample(input="The capital of France is")], solver=generate()),
    model=f"tl_bridge_vllm/{MODEL}",
    model_args={"capture": ["blocks.8.hook_out"]},
    max_tokens=4,
    log_dir=f"{run}/turns",
    display="none",
)[0]
turns = turn_activations(turn_log.samples[0])
print("model turns captured:", len(turns))
print("turn 0:", {k: tuple(v.shape) for k, v in turns[0].items()})

In [ ]:
# (9) Tool-aware generation. Same parser as the HF provider — re-exported from the
# vLLM provider for symmetry. For a tool-capable instruct model the provider renders
# tool schemas into the chat template; here we just show the parser:
from transformer_lens.model_bridge.sources.inspect.vllm_provider import _parse_tool_calls

calls = _parse_tool_calls('<tool_call>{"name": "add", "arguments": {"a": 2, "b": 2}}</tool_call>')
print("parsed tool calls:", [(c.function, c.arguments) for c in calls])

## Summary

- `boot_inspect(MODEL, provider="tl_bridge_vllm")` returns the same `RemoteBridge` as the
  HF provider, backed by vLLM. The structural self-check gates `resid_pre`/`resid_mid`
  (no Python-visible block-input hook under vLLM's fused execution).
- Residual / attn / mlp boundaries cross the Inspect boundary as the same wire envelope
  the HF provider emits, match `boot_transformers` to fp16 tolerance (next-token argmax
  agrees on served kinds), and convert to torch at the bridge.
- Full affine interventions (suppress / scale / add / set) flow through the per-hook
  scale+bias buffer swap on the worker.
- `tl_bridge_vllm/<model>` is also a normal Inspect model: eval-native generation,
  `capture_activations`, per-turn capture, tool-call parsing — same helpers, same wire
  format. Trade vs HF: gives up `resid_pre`/`resid_mid` and exact parity; gains vLLM's
  throughput (PagedAttention + continuous batching) for dataset-scale evals.